# IEMOCAP Data Analysis

Notebook này chỉ dùng để phân tích dữ liệu `AbstractTTS/IEMOCAP` và xem từng sample. Không load model/checkpoint.

In [2]:
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml
from IPython.display import Audio, display

from dataset import CANONICAL_LABELS, load_iemocap_splits
from features import describe_acoustic_cues, extract_acoustic_features

/opt/anaconda3/envs/speech/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Load config và dataset

In [5]:
CONFIG_PATH = Path("config.yaml")

with open(CONFIG_PATH, "r", encoding="utf-8") as handle:
    config = yaml.safe_load(handle)

sampling_rate = int(config.get("audio", {}).get("sampling_rate", 16000))
datasets = load_iemocap_splits(config)

print("sampling_rate:", sampling_rate)
for split, ds in datasets.items():
    print(split, len(ds), ds.column_names)

sampling_rate: 16000
train 7835 ['input_values', 'labels', 'emotion', 'transcript', 'speaking_rate', 'pitch_mean', 'pitch_std', 'rms', 'relative_db']
validation 979 ['input_values', 'labels', 'emotion', 'transcript', 'speaking_rate', 'pitch_mean', 'pitch_std', 'rms', 'relative_db']
test 980 ['input_values', 'labels', 'emotion', 'transcript', 'speaking_rate', 'pitch_mean', 'pitch_std', 'rms', 'relative_db']


In [6]:
import re

def parse_iemocap_id(utt_id: str):
    """
    Example:
    Ses01F_impro01_F000
    Ses02M_script03_1_M012
    """

    # Lấy session: Ses01, Ses02...
    m = re.search(r"(Ses\d{2})", utt_id)
    if not m:
        raise ValueError(f"Cannot parse session from: {utt_id}")
    session_id = m.group(1)

    parts = utt_id.split("_")

    # Speaker role thường nằm ở token cuối: F000 hoặc M012
    last = parts[-1]
    speaker_role = last[0]  # F or M

    if speaker_role not in ["F", "M"]:
        raise ValueError(f"Cannot parse speaker from: {utt_id}")

    # turn index
    turn_index = int(re.sub(r"\D", "", last))

    # dialogue id = bỏ phần utterance cuối
    dialogue_id = "_".join(parts[:-1])

    speaker_id = f"{session_id}_{speaker_role}"

    return {
        "session_id": session_id,
        "dialogue_id": dialogue_id,
        "speaker_role": speaker_role,
        "speaker_id": speaker_id,
        "turn_index": turn_index,
    }

## 2. Tổng quan phân bố label

In [ ]:
summary_rows = []
for split, ds in datasets.items():
    counts = Counter(ds["emotion"])
    total = len(ds)
    for label in CANONICAL_LABELS:
        summary_rows.append(
            {
                "split": split,
                "emotion": label,
                "count": counts.get(label, 0),
                "ratio": counts.get(label, 0) / total if total else 0.0,
            }
        )

label_df = pd.DataFrame(summary_rows)
label_df

NameError: name 'datasets' is not defined

In [ ]:
pivot = label_df.pivot(index="emotion", columns="split", values="count").fillna(0)
ax = pivot.plot(kind="bar", figsize=(8, 4), rot=0)
ax.set_title("Emotion label distribution")
ax.set_ylabel("count")
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()

## 3. Tạo bảng metadata để scan nhanh

In [ ]:
def build_metadata(split):
    rows = []
    ds = datasets[split]
    for idx, row in enumerate(ds):
        waveform = np.asarray(row["input_values"], dtype=np.float32)
        rows.append(
            {
                "split": split,
                "index": idx,
                "emotion": row["emotion"],
                "duration": len(waveform) / sampling_rate,
                "transcript": row.get("transcript", ""),
                "speaking_rate": row.get("speaking_rate"),
                "pitch_mean": row.get("pitch_mean"),
                "pitch_std": row.get("pitch_std"),
                "rms": row.get("rms"),
                "relative_db": row.get("relative_db"),
            }
        )
    return pd.DataFrame(rows)

metadata = pd.concat([build_metadata(split) for split in datasets.keys()], ignore_index=True)
metadata.head()

## 4. Thống kê duration và acoustic cues

In [ ]:
metadata.groupby(["split", "emotion"])[["duration", "speaking_rate", "pitch_mean", "pitch_std", "rms", "relative_db"]].describe().round(3)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
metadata.boxplot(column="duration", by="emotion", ax=axes[0])
axes[0].set_title("Duration by emotion")
axes[0].set_xlabel("emotion")
axes[0].set_ylabel("seconds")

metadata.boxplot(column="pitch_mean", by="emotion", ax=axes[1])
axes[1].set_title("Pitch mean by emotion")
axes[1].set_xlabel("emotion")
axes[1].set_ylabel("Hz")

plt.suptitle("")
plt.tight_layout()

## 5. Xem sample theo split/index

In [ ]:
def show_sample(split="train", index=0, play_audio=True):
    row = datasets[split][index]
    waveform = np.asarray(row["input_values"], dtype=np.float32)
    transcript = row.get("transcript", "")
    cues = extract_acoustic_features(waveform, sampling_rate, transcript=transcript, dataset_row=row)

    print("split:", split)
    print("index:", index)
    print("emotion:", row["emotion"])
    print("label_id:", row["labels"])
    print("duration:", round(len(waveform) / sampling_rate, 3), "seconds")
    print("transcript:", transcript)
    print("acoustic cues:", describe_acoustic_cues(cues))
    print("raw cues:", cues)

    if play_audio:
        display(Audio(waveform, rate=sampling_rate))

    fig, ax = plt.subplots(figsize=(12, 3))
    t = np.arange(len(waveform)) / sampling_rate
    ax.plot(t, waveform, linewidth=0.7)
    ax.set_title(f"Waveform: {split}[{index}] - {row['emotion']}")
    ax.set_xlabel("time (s)")
    ax.set_ylabel("amplitude")
    ax.grid(alpha=0.25)
    plt.tight_layout()

    return row

row = show_sample("train", 0, play_audio=True)

## 6. Xem sample theo emotion

In [ ]:
def list_samples(split="train", emotion=None, n=10, sort_by="index", ascending=True):
    df = metadata[metadata["split"] == split].copy()
    if emotion is not None:
        df = df[df["emotion"] == emotion]
    if sort_by in df.columns:
        df = df.sort_values(sort_by, ascending=ascending)
    return df.head(n)[["split", "index", "emotion", "duration", "transcript", "speaking_rate", "pitch_mean", "rms"]]

list_samples(split="train", emotion="sad", n=10)

In [ ]:
# Sau khi xem bảng trên, lấy index mong muốn và gọi show_sample.
show_sample("train", int(list_samples(split="train", emotion="sad", n=18).iloc[16]["index"]), play_audio=True)

## 7. Xem các sample dài/ngắn hoặc cue cực trị

In [ ]:
metadata.sort_values("duration", ascending=False).head(10)[["split", "index", "emotion", "duration", "transcript"]]

In [ ]:
metadata.sort_values("pitch_mean", ascending=False).head(10)[["split", "index", "emotion", "pitch_mean", "pitch_std", "transcript"]]

In [ ]:
metadata.sort_values("rms", ascending=False).head(10)[["split", "index", "emotion", "rms", "relative_db", "transcript"]]

## 8. Random sample browser

In [ ]:
def random_sample(split="train", emotion=None, seed=None, play_audio=True):
    rng = np.random.default_rng(seed)
    df = metadata[metadata["split"] == split]
    if emotion is not None:
        df = df[df["emotion"] == emotion]
    if len(df) == 0:
        raise ValueError("No samples match the query.")
    chosen = df.iloc[int(rng.integers(0, len(df)))]
    return show_sample(split, int(chosen["index"]), play_audio=play_audio)

random_sample(split="validation", emotion=None, seed=42, play_audio=True)